# ДЗ — Свой GPT за час: от датасета до HuggingFace Hub

> Если закрыли лекцию: это [Модуль 5](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-own-gpt/) курса «От нуля до своих агентов».

За ~90 минут вы **с нуля** обучаете крошечный трансформер на корпусе
Пушкина — двумя способами (char-level руками + BPE через `transformers.Trainer`)
— и заливаете обе модели на HuggingFace Hub.

**Главный урок:** «ChatGPT за час» вы не получите. Маленькая модель учит
**структуру** языка (ритм, окончания), а смысл приходит только от масштаба.

**Что от вас требуется:** запустить ноутбук целиком — он работает «из коробки»
(`Runtime → Run all`) — прочитать пояснения и сделать **задачи-доработки** в
конце: там вы меняете уже работающий код и смотрите, что меняется. Оба захода
(char-level и BPE) идут готовыми.

**Нужно до старта:** GPU T4 (Colab: `Runtime → T4 GPU`; Kaggle: `Accelerator → GPU`)
и HuggingFace-токен со scope `write` ([тут](https://huggingface.co/settings/tokens)).

**Как сдавать:** `Сохранить копию` → запустить все ячейки → выполнить
задачи-доработки → залить обе модели на Hub → прислать ссылку на профиль HF в
чат курса как `[Модуль 5, ДЗ] {ссылка}`.

**Время:** 90 минут (обучение на T4: ~25 мин Заход 1 + считаные минуты Заход 2).

## Как это работает: 5 коробок и стрелка назад

Прежде чем нырять в код — одна картинка, к которой стоит возвращаться. Любое
обучение LLM (хоть наш Пушкин, хоть Qwen) — это вот это:

```text
   текст  ──►  [ tokenizer ]  ──►  ids (числа)
                                       │
                                       ▼
                                  [  модель  ]  ──►  распределение
                                       ▲                над следующим токеном
                                       │                      │
                                       │                      ▼
                                       │            cross-entropy с правдой = loss
                                       └────────── backprop ──────────┘
```

Модель по последним токенам предсказывает **вероятности** следующего токена.
`loss` меряет, насколько промахнулась. `backprop` правит веса, чтобы в
следующий раз правда была вероятнее. Весь ноутбук ниже — это реализация этой
схемы, два раза (char-level и BPE).

## Шаг 0. Окружение и пакеты

In [ ]:
# В Colab/Kaggle большая часть уже стоит; accelerate нужен для Trainer (Заход 2).
!pip install -q torch transformers tokenizers datasets huggingface_hub "accelerate>=1.1.0"

import os, time, json, math, urllib.request
import torch, torch.nn as nn, torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu":
    print("ВНИМАНИЕ: GPU не найден. Код пойдёт, но на CPU медленно и результат слабее.")
    print("Включи T4: Colab -> Runtime -> Change runtime type -> T4 GPU; Kaggle -> Settings -> Accelerator -> GPU T4.")

## Шаг 1. Корпус Пушкина

Char-level датасет — словарь из отдельных символов. Public-domain текст
качаем готовым файлом.

In [ ]:
# Корпус Пушкина (public domain) — берём готовый train.txt из HF-датасета.
# ~0.86 МБ, 140 уникальных символов. Если сети нет — встроенный мини-фолбэк,
# чтобы Run All не падал (но на нём учиться особо нечему — нужен полный корпус).
CORPUS_URL = "https://huggingface.co/datasets/abobster/pushkin/resolve/main/train.txt"
os.makedirs("data", exist_ok=True)
path = "data/pushkin.txt"
if not os.path.exists(path):
    try:
        urllib.request.urlretrieve(CORPUS_URL, path)
        print("[ok] скачал корпус ->", path)
    except Exception as e:
        print("download failed:", e, "-> пишу мини-фолбэк")
        fallback = ("Я помню чудное мгновенье:\nПередо мной явилась ты,\n"
                    "Как мимолетное виденье,\nКак гений чистой красоты.\n") * 400
        open(path, "w", encoding="utf-8").write(fallback)

with open(path, encoding="utf-8") as f:
    text = f.read()
print(f"корпус: {len(text):,} символов, {len(set(text))} уникальных")
print("---- начало ----")
print(text[:200])

## Шаг 2. Char-level токенизатор и сплит train/val

In [ ]:
# Char-level токенизатор: словарь = уникальные символы корпуса.
# Никакого обучения токенизатора — просто перечисляем символы.
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s): return [stoi[c] for c in s]
def decode(ids): return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"vocab_size = {vocab_size}")
print(f"train: {len(train_data):,} токенов | val: {len(val_data):,}")

In [ ]:
# Покажем char-level токенизацию глазами: каждый символ -> своё число.
import matplotlib.pyplot as plt

def draw_token_row(ax, tokens, y, label, color="#DCE3F0", fixed_w=None, gap=0.25):
    # fixed_w задаёт одинаковую ширину всех боксов (для выравнивания рядов 1:1);
    # без него ширина пропорциональна длине куска (чтобы BPE-токен был «шире»).
    ax.text(-0.4, y, label, ha="right", va="center", fontsize=12, weight="bold")
    x = 0.0
    for t in tokens:
        w = fixed_w if fixed_w is not None else max(1.0, len(str(t)) * 0.85)
        ax.add_patch(plt.Rectangle((x, y - 0.4), w, 0.8, facecolor=color, edgecolor="#3355AA"))
        ax.text(x + w / 2.0, y, str(t), ha="center", va="center", fontsize=12)
        x += w + gap
    return x

word = "Привет"
fig, ax = plt.subplots(figsize=(10, 2.4))
# одинаковая ширина боксов -> каждый символ ровно над своим числом
e1 = draw_token_row(ax, list(word), 1.0, "символы", fixed_w=1.6)
e2 = draw_token_row(ax, encode(word), 0.0, "id (числа)", color="#F0E3D0", fixed_w=1.6)
ax.set_xlim(-3, max(e1, e2) + 1); ax.set_ylim(-0.7, 1.7); ax.axis("off")
ax.set_title(f"Char-level: '{word}' -> {len(word)} токенов  (словарь = {vocab_size} символов)")
plt.tight_layout(); plt.savefig("char_tokens.png", dpi=120); plt.show()
print("Модель видит только нижний ряд чисел — букв она не знает.")

## Шаг 3. Конфиг

Подстраивается под железо. На CPU — урезанный (чтобы хоть прошло), на T4 —
полный. Реальный результат — только на GPU.

In [ ]:
# Конфиг подстраивается под железо: на T4 — полный, на CPU — урезанный,
# чтобы ноутбук хотя бы прошёл. Для настоящего результата нужен GPU.
# (Для быстрой авто-проверки можно выставить переменную окружения M5_SMOKE=1.)
if os.environ.get("M5_SMOKE"):
    CFG = dict(block_size=64, n_layer=2, n_head=2, n_embd=128, batch_size=16, n_steps=60)
elif DEVICE == "cuda":
    CFG = dict(block_size=256, n_layer=4, n_head=4, n_embd=256, batch_size=64, n_steps=5000)
else:
    CFG = dict(block_size=128, n_layer=3, n_head=4, n_embd=128, batch_size=32, n_steps=1500)
print("config:", CFG)

## Шаг 4. Минимальный GPT (~80 строк) — собираем по частям

Не пугайся объёма: ниже — весь трансформер целиком. Понимать его построчно
прямо сейчас **не нужно** (детальный разбор внимания — в [модуле 5.5](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-5-llm-mental-model/)).
Достаточно увидеть три части — и мы пройдём их по очереди, с пояснением и
короткой проверкой после каждой:

1. **`CausalSelfAttention`** — «внимание»: каждый токен смотрит на предыдущие.
2. **`Block`** — один слой: внимание + маленькая нейросеть, аккуратно обёрнутые.
3. **`TinyGPT`** — всё вместе: числа → стопка блоков → вероятности следующего токена.

После каждой части идёт ячейка-проверка: запусти её и убедись, что кусок живой —
она печатает форму выхода. В конце соберём настоящую модель и посмотрим, **из
чего складываются её параметры**.

### Часть 1 — внимание (attention): «кто на кого смотрит»

Представь, что каждый токен в строке может **оглянуться назад** и спросить у
предыдущих: «кто из вас сейчас для меня важен?» — и подтянуть от них смысл. Вот
это «оглянуться и подтянуть» и есть **attention** (внимание).

Два правила, без формул:
- смотреть можно только **назад**, на уже прочитанное — будущее при обучении
  модель не подсматривает. За это отвечает «маска» (causal mask).
- «важность» — это числа, а не слова: чем сильнее запрос одного токена совпал
  с «ключом» другого, тем больше тот на него влияет.

Запусти ячейку — она определит механизм и тут же проверит его на игрушечном
входе: 5 токенов вошло, 5 вышло. Внимание **перемешивает смысл** между
позициями, а не меняет их число.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
        # causal-маска: токен i видит только токены <= i
        self.register_buffer("mask",
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.drop(att)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


# --- проверка: кусок живой? ---
_demo = CausalSelfAttention(n_embd=32, n_head=4, block_size=8)
_y = _demo(torch.zeros(1, 5, 32))   # batch=1, 5 токенов, 32 числа на токен
print("CausalSelfAttention ок: вход (1, 5, 32) -> выход", tuple(_y.shape))
print("длина та же (5) — внимание перемешивает смысл между токенами, не удлиняет последовательность")

### Часть 2 — блок (Block): внимание + «подумать» + страховка

Один **Block** — это один слой модели. Внутри две операции подряд:
1. **attention** (из части 1) — собрать контекст у соседей;
2. **MLP** — маленькая нейросеть, которая «обдумывает» собранное по каждому
   токену отдельно.

Плюс две инженерные страховки (тоже без формул):
- **residual** — запись `x + ...`: «оставляем оригинал и добавляем правку»,
  чтобы при обучении сигнал не терялся в глубине;
- **LayerNorm** — выравнивание чисел перед каждой операцией, чтобы они не
  разъезжались по масштабу.

Таких блоков в модели несколько — они и есть «глубина». Проверка снова покажет:
форма входа = форма выхода (поэтому блоки и можно ставить стопкой).

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


# --- проверка ---
_demo_block = Block(n_embd=32, n_head=4, block_size=8)
print("Block ок: вход (1, 5, 32) -> выход", tuple(_demo_block(torch.zeros(1, 5, 32)).shape))
print("форма сохраняется -> блоки складываются стопкой, это и есть глубина модели")

### Часть 3 — собираем GPT: числа → блоки → вероятности

Теперь всё вместе. **TinyGPT** делает ровно то, что на картинке «5 коробок» в
начале ноутбука:
- **эмбеддинги** — каждый id-токен превращается в вектор чисел (`tok_emb`), плюс
  отметка позиции в строке (`pos_emb`): «какое это число и где оно стоит»;
- **стопка блоков** — те самые слои внимания из частей 1–2;
- **голова** (`head`) — превращает итоговый вектор в **очки по всему словарю**:
  у какого токена какой шанс оказаться следующим.

Метод `generate` — это сэмплинг из этих вероятностей по одному токену (покрутим
его в TODO 2). А `forward` ещё и считает `loss`, если дать ему правильные ответы
(`targets`) — это понадобится в обучении.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size=256, n_layer=4, n_head=4, n_embd=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.ln_f(self.blocks(x))
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


# --- проверка на игрушечном словаре из 10 «символов» ---
_tiny = TinyGPT(vocab_size=10, block_size=8, n_layer=2, n_head=2, n_embd=32)
_logits, _ = _tiny(torch.zeros(1, 5, dtype=torch.long))
print("TinyGPT ок: выход", tuple(_logits.shape), "= для каждого из 5 токенов очки по всем 10 словам словаря")
print("дальше softmax превратит эти очки в вероятности следующего токена")

### Теперь по-настоящему: создаём модель и считаем параметры

Игрушки проверили — собираем модель нужного размера (из `CFG`) и смотрим,
**сколько в ней параметров**. Параметры — это все числа, которые обучение будет
крутить (веса). У нас их порядка пары миллионов; у GPT-4 — на пять порядков
больше при **той же** схеме. Ячейка печатает это число и заодно готовит
вспомогательные функции для обучения (`get_batch` — нарезает случайные окна,
`estimate_loss` — честно меряет loss на train и val).

In [ ]:
model = TinyGPT(vocab_size, block_size=CFG["block_size"], n_layer=CFG["n_layer"],
                n_head=CFG["n_head"], n_embd=CFG["n_embd"]).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
print(f"параметров: {sum(p.numel() for p in model.parameters()):,}")

def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - CFG["block_size"] - 1, (CFG["batch_size"],))
    x = torch.stack([src[i:i + CFG["block_size"]] for i in ix])
    y = torch.stack([src[i + 1:i + CFG["block_size"] + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def estimate_loss():
    model.train(False)
    out = {}
    for split in ["train", "val"]:
        losses = [model(*get_batch(split))[1].item() for _ in range(20)]
        out[split] = sum(losses) / len(losses)
    model.train(True)
    return out

### Куда уходят параметры — на одной картинке

Полезно увидеть, из чего складывается «вес» модели: сколько занимают эмбеддинги
(таблица токен→вектор), сколько — слои внимания, сколько — MLP внутри блоков,
сколько — голова. Ячейка ниже считает это по реальной модели и рисует.

In [ ]:
# Куда уходят параметры: разложим по группам и нарисуем.
import matplotlib.pyplot as plt
groups = {"эмбеддинги (tok+pos)": 0, "внимание (attn)": 0, "MLP в блоках": 0,
          "голова (head)": 0, "прочее (LayerNorm)": 0}
for name, p in model.named_parameters():
    n = p.numel()
    if name.startswith("tok_emb") or name.startswith("pos_emb"):
        groups["эмбеддинги (tok+pos)"] += n
    elif ".attn." in name:
        groups["внимание (attn)"] += n
    elif ".mlp." in name:
        groups["MLP в блоках"] += n
    elif name.startswith("head"):
        groups["голова (head)"] += n
    else:
        groups["прочее (LayerNorm)"] += n

total = sum(groups.values())
labels, vals = list(groups.keys()), list(groups.values())
plt.figure(figsize=(9, 4))
bars = plt.barh(labels, vals, color="#6A8CC7")
for bar, v in zip(bars, vals):
    plt.text(v, bar.get_y() + bar.get_height() / 2, f" {v:,} ({100*v/total:.0f}%)", va="center")
plt.gca().invert_yaxis()
plt.xlabel("число параметров"); plt.title(f"Из чего состоят {total:,} параметров модели")
plt.tight_layout(); plt.savefig("params_breakdown.png", dpi=120); plt.show()

top = max(groups, key=groups.get)
print(f"Больше всего параметров — в группе «{top}» ({100*groups[top]/total:.0f}%).")
print("Глубина (число блоков) и ширина (n_embd) растят модель быстрее всего;")
print("словарь добавляет к эмбеддингам и голове. У больших моделей всё то же — просто числа крупнее.")

## Шаг 5. Шаг обучения — сердце pretraining

Весь pretraining держится на четырёх строках в `train_step`: forward →
обнулить градиенты → backward → шаг оптимизатора. Это тот же цикл, что в
micrograd (4a) и nano-GPT (4b) — вырос только размер модели. Код ниже уже
написан и прокомментирован построчно: запусти и смотри, как `val` loss
поехал вниз (assert в цикле подтвердит, что обучение реально идёт). Покрутить
тут что-то руками — в задачах в конце ноутбука.

In [ ]:
def train_step(x, y):
    """Один шаг обучения — это и есть всё ядро pretraining, 4 строки.
    Те же, что в micrograd (4a) и nano-GPT (4b); вырос только размер модели."""
    _, loss = model(x, y)            # 1) forward: прогнали окно -> получили loss
    opt.zero_grad(set_to_none=True)  # 2) стёрли градиенты прошлого шага
    loss.backward()                  # 3) backward: посчитали, куда крутить каждый вес
    opt.step()                       # 4) шаг: чуть-чуть крутнули все веса
    return loss.item()               # число для логов

print("train_step готов: forward -> zero_grad -> backward -> step")

In [ ]:
t0 = time.time()
history = []
for step in range(CFG["n_steps"]):
    if step % max(1, CFG["n_steps"] // 20) == 0 or step == CFG["n_steps"] - 1:
        l = estimate_loss()
        history.append((step, l["train"], l["val"]))
        print(f"step {step:>5}  train {l['train']:.3f}  val {l['val']:.3f}  ({time.time()-t0:.0f}s)")
    x, y = get_batch("train")
    train_step(x, y)

print("готово. первый/последний val:", round(history[0][2], 3), "->", round(history[-1][2], 3))
assert history[-1][2] < history[0][2], "val loss не упал — что-то не так с train_step или данными"

### Кривая обучения

После того как TODO 1 заработал и loss поехал вниз — нарисуем это. Пунктир —
loss «случайной угадайки»; всё, что ниже, модель честно выучила.

In [ ]:
# Главный график урока: как падает loss = как модель учится.
import matplotlib.pyplot as plt
steps, tr, va = zip(*history)
rnd = math.log(vocab_size)  # loss «случайной угадайки» = ln(размер словаря)
plt.figure(figsize=(9, 5))
plt.plot(steps, tr, marker="o", label="train")
plt.plot(steps, va, marker="o", label="val")
plt.axhline(rnd, color="gray", ls="--", lw=1.2, label=f"случайная угадайка ≈ ln(vocab) = {rnd:.2f}")
plt.xlabel("шаг обучения"); plt.ylabel("loss (cross-entropy)")
plt.title("Вот оно учится: loss падает -> модель всё точнее угадывает следующий символ")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("loss_curve.png", dpi=120); plt.show()
print(f"старт ≈ {va[0]:.2f} (почти как случайная {rnd:.2f}) -> конец ≈ {va[-1]:.2f}")

## Шаг 6. Сэмплинг при трёх температурах

Метод `model.generate` уже написан, ячейка ниже зовёт его при трёх
температурах (0.1 / 0.8 / 1.5) — запусти и услышь разницу: 0.1 даёт повторы,
0.8 — «псевдопушкин», 1.5 — бред. Почему так — это семплинг из [модуля 5.5](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-5-llm-mental-model/).
Покрутить температуры самому — в задачах.

In [ ]:
prompt = "Я помню чудное мгновенье"
ctx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
for temp in [0.1, 0.8, 1.5]:
    out = model.generate(ctx, max_new_tokens=300, temperature=temp, top_k=40)
    print(f"\n--- temperature={temp} ---")
    print(decode(out[0].tolist()))

## Шаг 7. Публикуем char-level модель на Hub

Сначала логин (ячейка ниже), потом сохранение и заливка — код готовый,
запускается сам, когда ты залогинен. Это ваш **артефакт №1**:
`username/pushkin-nano`.

In [ ]:
# Логин в HuggingFace. Нужен токен со scope WRITE (huggingface.co/settings/tokens).
# В Colab/Kaggle удобнее положить токен в Secrets как HF_TOKEN.
from huggingface_hub import login, whoami
try:
    if os.environ.get("HF_TOKEN"):
        login(token=os.environ["HF_TOKEN"])
    HF_USER = whoami()["name"]
    HAVE_HF = True
    print("[ok] залогинен как", HF_USER)
except Exception as e:
    HAVE_HF = False
    print("HF не залогинен — push пропустится. Для сдачи ДЗ выполни notebook_login():")
    print("   from huggingface_hub import notebook_login; notebook_login()")

In [ ]:
REPO = "pushkin-nano"  # станет <твой-username>/pushkin-nano

# Сохраняем веса + словарь + конфиг (это и есть артефакт модели).
torch.save(model.state_dict(), "pytorch_model.bin")
json.dump(itos, open("vocab.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump({"model_type": "tiny-gpt-char", "vocab_size": vocab_size, **CFG},
          open("config.json", "w"))
assert all(os.path.exists(f) for f in ["pytorch_model.bin", "vocab.json", "config.json"])
print("[ok] файлы модели сохранены локально")

# Заливаем папку на Hub. Работает, когда ты залогинен (HAVE_HF=True).
if HAVE_HF:
    from huggingface_hub import HfApi, create_repo
    repo_id = f"{HF_USER}/{REPO}"
    create_repo(repo_id, exist_ok=True)
    HfApi().upload_folder(folder_path=".", repo_id=repo_id,
        allow_patterns=["pytorch_model.bin", "vocab.json", "config.json"])
    print("[ok] выложено:", "https://huggingface.co/" + repo_id)
else:
    print("HF не залогинен — push пропущен. Залогинься (ячейка выше) и перезапусти")
    print("эту ячейку, чтобы выложить артефакт №1.")

## Заход 2: BPE + transformers.Trainer (Build-twice)

Тот же датасет и цель — но через готовые HF-абстракции. Вы только что
написали train loop руками; теперь видно, что `Trainer` делает то же самое
в ~30 строк. Это **артефакт №2**: `username/pushkin-nano-bpe`.
Код готовый — просто запускайте.

In [ ]:
# Учим BPE-токенизатор на том же корпусе. vocab_size=1024 — итоговый размер словаря.
from tokenizers import ByteLevelBPETokenizer
os.makedirs("pushkin-bpe", exist_ok=True)
bpe = ByteLevelBPETokenizer()
bpe.train(files=["data/pushkin.txt"], vocab_size=1024, min_frequency=2,
          special_tokens=["<|endoftext|>"])
# ВАЖНО: сохраняем как единый tokenizer.json и грузим через PreTrainedTokenizerFast.
# (Старый путь GPT2TokenizerFast(vocab_file=, merges_file=) в свежих версиях
#  transformers даёт пустой словарь — vocab_size 0.)
bpe.save("pushkin-bpe/tokenizer.json")

from transformers import PreTrainedTokenizerFast
tok = PreTrainedTokenizerFast(
    tokenizer_file="pushkin-bpe/tokenizer.json",
    bos_token="<|endoftext|>", eos_token="<|endoftext|>",
    pad_token="<|endoftext|>", unk_token="<|endoftext|>")

sample = "Я помню чудное мгновенье"
print("char-level:", len(sample), "токенов | BPE-1024:", len(tok.encode(sample)), "токенов")
print("BPE словарь:", tok.vocab_size)

**Зачем нужен BPE — на одной картинке.** Те же слова, но токенов
меньше: BPE склеивает частые куски символов. Меньше токенов → за тот же
compute модель «прочитывает» больше текста.

In [ ]:
# Зачем нужен BPE — видно на одной фразе: те же слова, меньше токенов.
import matplotlib.pyplot as plt
sample = "Я помню чудное мгновенье"
# куски BPE берём по offset'ам в исходной строке — так подписи чистые,
# несмотря на байтовую «кухню» BPE внутри
enc = tok(sample, return_offsets_mapping=True)
bpe_pieces = [sample[s:e] for s, e in enc["offset_mapping"] if e > s]
char_pieces = list(sample)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 4.4),
                               gridspec_kw={"height_ratios": [2, 1]})
e1 = draw_token_row(ax1, char_pieces, 1.0, f"char ({len(char_pieces)})", color="#F0E3D0")
e2 = draw_token_row(ax1, bpe_pieces, 0.0, f"BPE ({len(bpe_pieces)})", color="#D0EAD8")
ax1.set_xlim(-3, max(e1, e2) + 1); ax1.set_ylim(-0.7, 1.7); ax1.axis("off")
ax1.set_title(f"«{sample}» — BPE склеивает частые куски, токенов меньше")

ax2.barh(["char-level\n(словарь ~140)", "BPE-1024"],
         [len(char_pieces), len(bpe_pieces)], color=["#E0A060", "#60A878"])
for i, v in enumerate([len(char_pieces), len(bpe_pieces)]):
    ax2.text(v + 0.2, i, str(v), va="center", weight="bold")
ax2.set_xlabel("длина фразы в токенах (меньше = больше текста за тот же compute)")
ax2.invert_yaxis()
plt.tight_layout(); plt.savefig("tokenization_compare.png", dpi=120); plt.show()

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import Dataset

eos_id = tok.convert_tokens_to_ids("<|endoftext|>")
cfg = GPT2Config(vocab_size=tok.vocab_size, n_positions=CFG["block_size"],
                 n_embd=CFG["n_embd"], n_layer=CFG["n_layer"], n_head=CFG["n_head"],
                 bos_token_id=eos_id, eos_token_id=eos_id)
model_bpe = GPT2LMHeadModel(cfg).to(DEVICE)
print(f"параметров (BPE): {model_bpe.num_parameters():,}")

# Режем корпус на блоки длины block_size
ids = tok.encode(text)
bs = CFG["block_size"]
chunks = [ids[i:i + bs] for i in range(0, len(ids) - bs, bs)]
ds = Dataset.from_dict({"input_ids": chunks, "labels": chunks})

args = TrainingArguments(
    output_dir="pushkin-bpe-out", num_train_epochs=1,
    per_device_train_batch_size=max(8, CFG["batch_size"] // 2),
    learning_rate=3e-4, logging_steps=100, save_strategy="no",
    report_to="none", fp16=(DEVICE == "cuda"))
Trainer(model=model_bpe, args=args, train_dataset=ds).train()

In [ ]:
inp = tok("Я помню чудное мгновенье", return_tensors="pt").to(model_bpe.device)
gen = model_bpe.generate(**inp, max_new_tokens=200, do_sample=True,
                         temperature=0.8, top_k=40, pad_token_id=eos_id)
print(tok.decode(gen[0]))

if HAVE_HF:
    model_bpe.push_to_hub(f"{HF_USER}/pushkin-nano-bpe")
    tok.push_to_hub(f"{HF_USER}/pushkin-nano-bpe")
    print("выложено: https://huggingface.co/" + f"{HF_USER}/pushkin-nano-bpe")
else:
    print("HF не залогинен — push BPE-модели пропущен.")

## Задачи — доработайте рабочий код

Ноутбук уже отработал целиком. Теперь учимся, **меняя готовое** и наблюдая
эффект. Правьте прямо в ячейках выше (или скопируйте их вниз):

1. **Температуры.** В сэмплинге (Шаг 6) добавьте в список `0.5` и `2.0`. Где
   начинается «псевдопушкин», а где — бред? Запишите словами.
2. **Свой prompt.** Смените `prompt` на свою строку и посмотрите продолжение.
3. **Дольше учим.** Увеличьте `n_steps` в `CFG` в 1.5–2 раза и перезапустите
   обучение. Loss продолжает падать или вышел на плато? (сравните по графику)
4. **Больше модель.** Поднимите `n_layer` или `n_embd` в `CFG`. Как изменилось
   число параметров (разбивка в Шаге 4) и итоговый `val` loss?
5. **(advanced) top-p.** В `TinyGPT.generate` уже есть `top_k`. Добавьте рядом
   nucleus-сэмплинг (`top_p`): оставляйте минимальный набор токенов с суммарной
   вероятностью ≥ p. Сравните вывод с `top_k` при той же температуре.

Каждая задача — правка рабочего кода, а не пустой лист. По каждой запишите
короткий вывод в markdown-ячейку.

## Что вы сделали и чек перед сдачей

Вы прошли полный pretraining-пайплайн дважды: char-level руками и BPE через
Trainer. Loss двух заходов **сравнивать нельзя** — словари разные; сравнивайте
качество сэмплов на слух.

**Чек перед сдачей:**
- [ ] Ноутбук прогнан целиком (`Run all`) без правок — `val` loss в Заходе 1 упал.
- [ ] Сэмплы при temperature 0.1 / 0.8 / 1.5 распечатаны.
- [ ] Сделаны задачи-доработки (мин. 3 из 5) с короткими выводами.
- [ ] На HF Hub лежат `username/pushkin-nano` (char) и `username/pushkin-nano-bpe` (BPE).
- [ ] Ссылка на профиль HF прислана как `[Модуль 5, ДЗ] {ссылка}`.

Дальше — [Модуль 5.5: Как LLM думает](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-5-llm-mental-model/).